## Working with AWS S3

Welcome back! In the previous lesson, you learned the fundamentals of AWS and the boto3 Universal Pattern — importing the SDK, creating a client, performing an operation, and processing the response. Now, we are moving on to one of the most popular AWS services: Amazon S3.

Amazon S3 (Simple Storage Service) is a service that lets you store and retrieve any amount of data, at any time, from anywhere on the web. It is widely used for storing files, backups, images, and much more.

In this lesson, you will learn how to:

* Make sure an S3 bucket exists (and create one if it doesn't)
* Upload a file to S3
* List files in your S3 bucket
* Download a file from S3

By the end of this lesson, you will be able to interact with S3 using Python code, which is a key skill for many cloud development tasks.

---

## Recall: Connecting to AWS S3 with boto3

Before we start working with S3, let's quickly remind ourselves how to connect to AWS services using the boto3 library in Python. You learned about this in the previous lesson, but here's a short reminder.

First, make sure you have boto3 installed. If you're working on your own machine, you can install it using pip:

```shell
pip install boto3
```

On CodeSignal, boto3 is usually pre-installed for you, so you can skip this step.

To use boto3, you need to have AWS credentials set up. On CodeSignal, these are usually pre-configured for you. On your own machine, you would set up credentials using the AWS CLI or environment variables.

To connect to S3, you create a client like this:

```python
import boto3

s3 = boto3.client('s3')
```

Here, `boto3.client('s3')` creates a client object that lets you interact with S3. You will use this `s3` object for all your S3 operations in this lesson.

---

## Ensuring Your S3 Bucket Exists

Before you can upload or download files, you need a bucket. A bucket is like a folder in S3 where your files (called "objects") are stored. Bucket names must be globally unique across all AWS users.

Let's see how to check if a bucket exists and create it if it doesn't.

First, you need to decide on a bucket name. In the example code, we use a unique name with a random part to avoid conflicts:

```python
import os
import uuid

BUCKET = os.environ.get("BUCKET") or f"aws-s3-practice-bucket-{str(uuid.uuid4())[:8]}"
```

* `os.environ.get("BUCKET")` checks if there is a `BUCKET` environment variable set.
* If not, it creates a new name like `aws-s3-practice-bucket-1a2b3c4d` using a random string.

Now, let's check if the bucket exists and create it if needed:

```python
from botocore.exceptions import ClientError

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        return
    except ClientError as e:
        code = (e.response.get("Error") or {}).get("Code")
        # If the bucket does NOT exist, create it
        if code in ("404", "NoSuchBucket"):
            client_region = s3.meta.region_name or boto3.Session().region_name or "us-east-1"
            params = {"Bucket": bucket}
            if client_region != "us-east-1":
                params["CreateBucketConfiguration"] = {"LocationConstraint": client_region}
            s3.create_bucket(**params)
            return
        # If we get 403, the bucket likely exists but you don't own it
        if code in ("403", "AccessDenied"):
            raise SystemExit(
                f"Bucket '{bucket}' exists but you do not have access. "
                "Set BUCKET to a globally-unique bucket name you own."
            )
        # Otherwise, re-raise unexpected errors
        raise
```

**Explanation:**

* `s3.head_bucket(Bucket=bucket)` checks if the bucket exists and you have access.
* If the bucket does not exist (404), it creates the bucket in your region.
* If you get a 403 error, the bucket exists but you don't own it, so you should pick a different name.
* Any other errors are re-raised.

Example output if the bucket is created:

```text
Bucket 'aws-s3-practice-bucket-1a2b3c4d' created.
```

---

## Uploading a File to S3

Now that you have a bucket, let's upload a file to it. First, you need a file to upload. Let's create a simple text file and then upload it.

```python
KEY = "examples/sample.txt"
LOCAL_UPLOAD = "sample.txt"

def upload(bucket):
    with open(LOCAL_UPLOAD, "w") as f:
        f.write("Hello from boto3 S3!\n")
    s3.upload_file(LOCAL_UPLOAD, bucket, KEY)
    print(f"Uploaded to s3://{bucket}/{KEY}")
```

**Explanation:**

* `with open(LOCAL_UPLOAD, "w") as f:` creates a new file called `sample.txt` and opens it for writing.
* `f.write("Hello from boto3 S3!\n")` writes a simple message to the file.
* `s3.upload_file(LOCAL_UPLOAD, bucket, KEY)` uploads the file to your S3 bucket under the key (path) `examples/sample.txt`.
* The `KEY` is like the file path inside your bucket.

Example output:

```text
Uploaded to s3://aws-s3-practice-bucket-1a2b3c4d/examples/sample.txt
```

---

## Listing Files in Your S3 Bucket

After uploading, you might want to see what files are in your bucket. You can list objects in a specific folder (called a "prefix" in S3).

```python
def list_objects(bucket):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix="examples/")
    for obj in resp.get("Contents", []):
        print(obj["Key"], obj["Size"])
```

**Explanation:**

* `s3.list_objects_v2(Bucket=bucket, Prefix="examples/")` lists all objects in the `examples/` folder of your bucket.
* The response contains a list of objects under the `"Contents"` key.
* For each object, we print its key (file path) and size in bytes.

Example output:

```text
examples/sample.txt 20
```

This tells you that there is a file called `examples/sample.txt` and its size is 20 bytes.

---

## Downloading a File from S3

Finally, let's download the file you uploaded back to your local machine. We'll save it in a folder called `downloads`.

```python
import os

DOWNLOAD_DIR = "downloads"

def download(bucket):
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    out = os.path.join(DOWNLOAD_DIR, "downloaded.txt")
    s3.download_file(bucket, KEY, out)
    print(f"Downloaded to {out}")
```

**Explanation:**

* `os.makedirs(DOWNLOAD_DIR, exist_ok=True)` creates the `downloads` directory if it doesn't already exist.
* `out = os.path.join(DOWNLOAD_DIR, "downloaded.txt")` sets the path for the downloaded file.
* `s3.download_file(bucket, KEY, out)` downloads the file from S3 to your local directory.

Example output:

```text
Downloaded to downloads/downloaded.txt
```

---

## Summary and What's Next

In this lesson, you learned how to:

* Make sure an S3 bucket exists (and create one if needed)
* Upload a file to S3
* List files in your S3 bucket
* Download a file from S3

These are the basic building blocks for working with files in AWS S3 using Python. Up next, you'll get to practice these steps yourself in hands-on exercises. This will help you get comfortable with S3 operations and prepare you for more advanced AWS development tasks. Good luck!

## Customizing S3 File Upload Operations

Now that you understand how S3 file uploads work, let's practice customizing the upload process. You'll modify an existing upload function to change both the file content and where it gets stored in S3.

Your objective is to update the code so that it creates a welcome message instead of a simple greeting and stores it in a different folder structure. Look for the TODO comments in the code — they will guide you to the exact lines that need changes.

You need to:

* Change the S3 key from `examples/sample.txt` to `documents/welcome.txt`
* Update the file content to `Welcome to AWS S3 development!\n`
* Update the list function to look in the correct folder

After making these changes, run the script and observe how your file now appears under the `documents/` folder in S3. This will help you understand how S3 keys work as file paths within your bucket.

```python
import os, boto3
from botocore.exceptions import ClientError

s3 = boto3.client('s3')

import uuid

BUCKET = os.environ.get("BUCKET") or f"aws-s3-practice-bucket-{str(uuid.uuid4())[:8]}"
# TODO: Change the KEY from "examples/sample.txt" to "documents/welcome.txt"
KEY = "examples/sample.txt"
LOCAL_UPLOAD = "sample.txt"
DOWNLOAD_DIR = "downloads"

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        return
    except ClientError as e:
        code = (e.response.get("Error") or {}).get("Code")
        # If the bucket does NOT exist, create it (in the client's region)
        if code in ("404", "NoSuchBucket"):
            client_region = s3.meta.region_name or boto3.Session().region_name or "us-east-1"
            params = {"Bucket": bucket}
            if client_region != "us-east-1":
                params["CreateBucketConfiguration"] = {"LocationConstraint": client_region}
            s3.create_bucket(**params)
            return
        # If we get 403, the bucket likely exists but you don't own it
        if code in ("403", "AccessDenied"):
            raise SystemExit(
                f"Bucket '{bucket}' exists but you do not have access. "
                "Set BUCKET to a globally-unique bucket name you own."
            )
        # Otherwise, re-raise unexpected errors
        raise

def upload(bucket):
    with open(LOCAL_UPLOAD, "w") as f:
        # TODO: Change the message to "Welcome to AWS S3 development!\n"
        f.write("Hello from boto3 S3!\n")
    s3.upload_file(LOCAL_UPLOAD, bucket, KEY)
    print(f"Uploaded to s3://{bucket}/{KEY}")

def list_objects(bucket):
    # TODO: Update the Prefix to "documents/" to match your new key
    resp = s3.list_objects_v2(Bucket=bucket, Prefix="examples/")
    for obj in resp.get("Contents", []):
        print(obj["Key"], obj["Size"])

def download(bucket):
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    out = os.path.join(DOWNLOAD_DIR, "downloaded.txt")
    s3.download_file(bucket, KEY, out)
    print(f"Downloaded to {out}")

if __name__ == "__main__":
    if not BUCKET:
        raise SystemExit("Set BUCKET environment variable to an existing bucket you own.")
    ensure_bucket(BUCKET)
    upload(BUCKET)
    list_objects(BUCKET)
    download(BUCKET)
```

Here is the completed code with the key, file content, and list prefix all updated:

```python
import os, boto3
from botocore.exceptions import ClientError

s3 = boto3.client('s3')

import uuid

BUCKET = os.environ.get("BUCKET") or f"aws-s3-practice-bucket-{str(uuid.uuid4())[:8]}"
KEY = "documents/welcome.txt"
LOCAL_UPLOAD = "sample.txt"
DOWNLOAD_DIR = "downloads"

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        return
    except ClientError as e:
        code = (e.response.get("Error") or {}).get("Code")
        # If the bucket does NOT exist, create it (in the client's region)
        if code in ("404", "NoSuchBucket"):
            client_region = s3.meta.region_name or boto3.Session().region_name or "us-east-1"
            params = {"Bucket": bucket}
            if client_region != "us-east-1":
                params["CreateBucketConfiguration"] = {"LocationConstraint": client_region}
            s3.create_bucket(**params)
            return
        # If we get 403, the bucket likely exists but you don't own it
        if code in ("403", "AccessDenied"):
            raise SystemExit(
                f"Bucket '{bucket}' exists but you do not have access. "
                "Set BUCKET to a globally-unique bucket name you own."
            )
        # Otherwise, re-raise unexpected errors
        raise

def upload(bucket):
    with open(LOCAL_UPLOAD, "w") as f:
        f.write("Welcome to AWS S3 development!\n")
    s3.upload_file(LOCAL_UPLOAD, bucket, KEY)
    print(f"Uploaded to s3://{bucket}/{KEY}")

def list_objects(bucket):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix="documents/")
    for obj in resp.get("Contents", []):
        print(obj["Key"], obj["Size"])

def download(bucket):
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    out = os.path.join(DOWNLOAD_DIR, "downloaded.txt")
    s3.download_file(bucket, KEY, out)
    print(f"Downloaded to {out}")

if __name__ == "__main__":
    if not BUCKET:
        raise SystemExit("Set BUCKET environment variable to an existing bucket you own.")
    ensure_bucket(BUCKET)
    upload(BUCKET)
    list_objects(BUCKET)
    download(BUCKET)
```

## Fixing S3 Bucket Creation Logic

Nice work learning about S3 bucket creation and file operations! Now it's time to put your debugging skills to the test with a common issue that occurs when working with AWS error handling.

You have a Python script that should create an S3 bucket, upload a file, list the contents, and download the file back. However, there's a bug in the `ensure_bucket` function that's causing the script to crash when it tries to create a new bucket.

The problem is in the error handling logic — the code is checking for the wrong error code when determining whether a bucket exists. When S3 can't find a bucket, it returns a `"404"` error code (meaning "not found"), but the current code is looking for a different error code entirely.

Your task is to find the incorrect error code in the `ensure_bucket` function and fix it. Look for the line that checks error codes and change it so the function properly recognizes when a bucket is missing.

Once you fix this bug, the script will run smoothly and demonstrate the complete S3 workflow from bucket creation to file download.

```python
import os, boto3
from botocore.exceptions import ClientError

s3 = boto3.client('s3')

import uuid

BUCKET = os.environ.get("BUCKET") or f"aws-s3-practice-bucket-{str(uuid.uuid4())[:8]}"
KEY = "examples/sample.txt"
LOCAL_UPLOAD = "sample.txt"
DOWNLOAD_DIR = "downloads"

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        return
    except ClientError as e:
        code = (e.response.get("Error") or {}).get("Code")
        # If the bucket does NOT exist, create it (in the client's region)
        if code in ("500", "NoSuchBucket"):
            client_region = s3.meta.region_name or boto3.Session().region_name or "us-east-1"
            params = {"Bucket": bucket}
            if client_region != "us-east-1":
                params["CreateBucketConfiguration"] = {"LocationConstraint": client_region}
            s3.create_bucket(**params)
            return
        # If we get 403, the bucket likely exists but you don't own it
        if code in ("403", "AccessDenied"):
            raise SystemExit(
                f"Bucket '{bucket}' exists but you do not have access. "
                "Set BUCKET to a globally-unique bucket name you own."
            )
        # Otherwise, re-raise unexpected errors
        raise

def upload(bucket):
    with open(LOCAL_UPLOAD, "w") as f:
        f.write("Hello from boto3 S3!\n")
    s3.upload_file(LOCAL_UPLOAD, bucket, KEY)
    print(f"Uploaded to s3://{bucket}/{KEY}")

def list_objects(bucket):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix="examples/")
    for obj in resp.get("Contents", []):
        print(obj["Key"], obj["Size"])

def download(bucket):
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    out = os.path.join(DOWNLOAD_DIR, "downloaded.txt")
    s3.download_file(bucket, KEY, out)
    print(f"Downloaded to {out}")

if __name__ == "__main__":
    if not BUCKET:
        raise SystemExit("Set BUCKET environment variable to an existing bucket you own.")
    ensure_bucket(BUCKET)
    upload(BUCKET)
    list_objects(BUCKET)
    download(BUCKET)
```

Here is the fixed code with the error code corrected from `"500"` to `"404"`:

```python
import os, boto3
from botocore.exceptions import ClientError

s3 = boto3.client('s3')

import uuid

BUCKET = os.environ.get("BUCKET") or f"aws-s3-practice-bucket-{str(uuid.uuid4())[:8]}"
KEY = "examples/sample.txt"
LOCAL_UPLOAD = "sample.txt"
DOWNLOAD_DIR = "downloads"

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        return
    except ClientError as e:
        code = (e.response.get("Error") or {}).get("Code")
        # If the bucket does NOT exist, create it (in the client's region)
        if code in ("404", "NoSuchBucket"):
            client_region = s3.meta.region_name or boto3.Session().region_name or "us-east-1"
            params = {"Bucket": bucket}
            if client_region != "us-east-1":
                params["CreateBucketConfiguration"] = {"LocationConstraint": client_region}
            s3.create_bucket(**params)
            return
        # If we get 403, the bucket likely exists but you don't own it
        if code in ("403", "AccessDenied"):
            raise SystemExit(
                f"Bucket '{bucket}' exists but you do not have access. "
                "Set BUCKET to a globally-unique bucket name you own."
            )
        # Otherwise, re-raise unexpected errors
        raise

def upload(bucket):
    with open(LOCAL_UPLOAD, "w") as f:
        f.write("Hello from boto3 S3!\n")
    s3.upload_file(LOCAL_UPLOAD, bucket, KEY)
    print(f"Uploaded to s3://{bucket}/{KEY}")

def list_objects(bucket):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix="examples/")
    for obj in resp.get("Contents", []):
        print(obj["Key"], obj["Size"])

def download(bucket):
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    out = os.path.join(DOWNLOAD_DIR, "downloaded.txt")
    s3.download_file(bucket, KEY, out)
    print(f"Downloaded to {out}")

if __name__ == "__main__":
    if not BUCKET:
        raise SystemExit("Set BUCKET environment variable to an existing bucket you own.")
    ensure_bucket(BUCKET)
    upload(BUCKET)
    list_objects(BUCKET)
    download(BUCKET)
```

## Completing S3 Upload Function Parameters

Excellent work customizing your S3 upload operations! Now, let's focus on completing a partially written upload function to understand how the `s3.upload_file()` method works.

You have a script that handles the complete S3 workflow, but there's an issue in the upload function. The function creates a local file correctly, but the `s3.upload_file()` call is missing some required parameters.

The `s3.upload_file()` method needs three parameters in this order:

* The local file path (source)
* The bucket name (destination bucket)
* The S3 key (destination path in the bucket)

Currently, the function only passes the bucket parameter. You'll need to add the missing `LOCAL_UPLOAD` and `KEY` variables, which are already defined at the top of the file.

Once you complete the upload function, run the script to see your file successfully uploaded and listed in your S3 bucket!

```python
import os, boto3
from botocore.exceptions import ClientError

s3 = boto3.client('s3')

import uuid

BUCKET = os.environ.get("BUCKET") or f"aws-s3-practice-bucket-{str(uuid.uuid4())[:8]}"
KEY = "examples/sample.txt"
LOCAL_UPLOAD = "sample.txt"
DOWNLOAD_DIR = "downloads"

def upload(bucket):
    with open(LOCAL_UPLOAD, "w") as f:
        f.write("Hello from boto3 S3!\n")
    # TODO: Complete the upload_file call by adding the missing LOCAL_UPLOAD and KEY parameters
    s3.upload_file(bucket)
    print(f"Uploaded to s3://{bucket}/{KEY}")

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        return
    except ClientError as e:
        code = (e.response.get("Error") or {}).get("Code")
        # If the bucket does NOT exist, create it (in the client's region)
        if code in ("404", "NoSuchBucket"):
            client_region = s3.meta.region_name or boto3.Session().region_name or "us-east-1"
            params = {"Bucket": bucket}
            if client_region != "us-east-1":
                params["CreateBucketConfiguration"] = {"LocationConstraint": client_region}
            s3.create_bucket(**params)
            return
        # If we get 403, the bucket likely exists but you don't own it
        if code in ("403", "AccessDenied"):
            raise SystemExit(
                f"Bucket '{bucket}' exists but you do not have access. "
                "Set BUCKET to a globally-unique bucket name you own."
            )
        # Otherwise, re-raise unexpected errors
        raise

def list_objects(bucket):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix="examples/")
    for obj in resp.get("Contents", []):
        print(obj["Key"], obj["Size"])

def download(bucket):
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    out = os.path.join(DOWNLOAD_DIR, "downloaded.txt")
    s3.download_file(bucket, KEY, out)
    print(f"Downloaded to {out}")

if __name__ == "__main__":
    if not BUCKET:
        raise SystemExit("Set BUCKET environment variable to an existing bucket you own.")
    ensure_bucket(BUCKET)
    upload(BUCKET)
    list_objects(BUCKET)
    download(BUCKET)
```

Here is the completed code with the missing `LOCAL_UPLOAD` and `KEY` parameters added to `s3.upload_file()`:

```python
import os, boto3
from botocore.exceptions import ClientError

s3 = boto3.client('s3')

import uuid

BUCKET = os.environ.get("BUCKET") or f"aws-s3-practice-bucket-{str(uuid.uuid4())[:8]}"
KEY = "examples/sample.txt"
LOCAL_UPLOAD = "sample.txt"
DOWNLOAD_DIR = "downloads"

def upload(bucket):
    with open(LOCAL_UPLOAD, "w") as f:
        f.write("Hello from boto3 S3!\n")
    s3.upload_file(LOCAL_UPLOAD, bucket, KEY)
    print(f"Uploaded to s3://{bucket}/{KEY}")

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        return
    except ClientError as e:
        code = (e.response.get("Error") or {}).get("Code")
        # If the bucket does NOT exist, create it (in the client's region)
        if code in ("404", "NoSuchBucket"):
            client_region = s3.meta.region_name or boto3.Session().region_name or "us-east-1"
            params = {"Bucket": bucket}
            if client_region != "us-east-1":
                params["CreateBucketConfiguration"] = {"LocationConstraint": client_region}
            s3.create_bucket(**params)
            return
        # If we get 403, the bucket likely exists but you don't own it
        if code in ("403", "AccessDenied"):
            raise SystemExit(
                f"Bucket '{bucket}' exists but you do not have access. "
                "Set BUCKET to a globally-unique bucket name you own."
            )
        # Otherwise, re-raise unexpected errors
        raise

def list_objects(bucket):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix="examples/")
    for obj in resp.get("Contents", []):
        print(obj["Key"], obj["Size"])

def download(bucket):
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    out = os.path.join(DOWNLOAD_DIR, "downloaded.txt")
    s3.download_file(bucket, KEY, out)
    print(f"Downloaded to {out}")

if __name__ == "__main__":
    if not BUCKET:
        raise SystemExit("Set BUCKET environment variable to an existing bucket you own.")
    ensure_bucket(BUCKET)
    upload(BUCKET)
    list_objects(BUCKET)
    download(BUCKET)
```

## Filtering and Formatting S3 Object Lists

Perfect! You've mastered uploading files and handling S3 parameters. Now, let's learn how to process and filter the results when listing objects in your S3 bucket.

You have a working S3 script that uploads a file to the `documents/` folder, but the `list_objects` function needs some improvements. Currently, it shows all files regardless of size and displays the output in a basic format.

Your task is to make the listing more useful by adding filtering and better formatting:

* Update the folder location from `"examples/"` to `"documents/"`
* Only show files that are larger than 10 bytes
* Add the word "bytes" after the file size for better readability

Look for the TODO comments in the code to find the exact lines that need changes. This exercise will teach you how to process S3 API responses and apply filtering logic to make your applications more professional and user-friendly.

```python
import os, boto3
from botocore.exceptions import ClientError

s3 = boto3.client('s3')

import uuid

BUCKET = os.environ.get("BUCKET") or f"aws-s3-practice-bucket-{str(uuid.uuid4())[:8]}"
KEY = "documents/sample.txt"
LOCAL_UPLOAD = "sample.txt"
DOWNLOAD_DIR = "downloads"

def list_objects(bucket):
    # TODO: Change the Prefix from "examples/" to "documents/" to match where your file is stored
    resp = s3.list_objects_v2(Bucket=bucket, Prefix="examples/")
    for obj in resp.get("Contents", []):
        # TODO: Add a condition to only show files larger than 10 bytes
        # TODO: Update the print statement to include "bytes" after the size number
        print(obj["Key"], obj["Size"])

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        return
    except ClientError as e:
        code = (e.response.get("Error") or {}).get("Code")
        # If the bucket does NOT exist, create it (in the client's region)
        if code in ("404", "NoSuchBucket"):
            client_region = s3.meta.region_name or boto3.Session().region_name or "us-east-1"
            params = {"Bucket": bucket}
            if client_region != "us-east-1":
                params["CreateBucketConfiguration"] = {"LocationConstraint": client_region}
            s3.create_bucket(**params)
            return
        # If we get 403, the bucket likely exists but you don't own it
        if code in ("403", "AccessDenied"):
            raise SystemExit(
                f"Bucket '{bucket}' exists but you do not have access. "
                "Set BUCKET to a globally-unique bucket name you own."
            )
        # Otherwise, re-raise unexpected errors
        raise

def upload(bucket):
    with open(LOCAL_UPLOAD, "w") as f:
        f.write("Hello from boto3 S3!\n")
    s3.upload_file(LOCAL_UPLOAD, bucket, KEY)
    print(f"Uploaded to s3://{bucket}/{KEY}")

def download(bucket):
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    out = os.path.join(DOWNLOAD_DIR, "downloaded.txt")
    s3.download_file(bucket, KEY, out)
    print(f"Downloaded to {out}")

if __name__ == "__main__":
    if not BUCKET:
        raise SystemExit("Set BUCKET environment variable to an existing bucket you own.")
    ensure_bucket(BUCKET)
    upload(BUCKET)
    list_objects(BUCKET)
    download(BUCKET)
```

Here is the completed code with the prefix updated, a size filter added, and "bytes" appended to the output:

```python
import os, boto3
from botocore.exceptions import ClientError

s3 = boto3.client('s3')

import uuid

BUCKET = os.environ.get("BUCKET") or f"aws-s3-practice-bucket-{str(uuid.uuid4())[:8]}"
KEY = "documents/sample.txt"
LOCAL_UPLOAD = "sample.txt"
DOWNLOAD_DIR = "downloads"

def list_objects(bucket):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix="documents/")
    for obj in resp.get("Contents", []):
        if obj["Size"] > 10:
            print(obj["Key"], f'{obj["Size"]} bytes')

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        return
    except ClientError as e:
        code = (e.response.get("Error") or {}).get("Code")
        # If the bucket does NOT exist, create it (in the client's region)
        if code in ("404", "NoSuchBucket"):
            client_region = s3.meta.region_name or boto3.Session().region_name or "us-east-1"
            params = {"Bucket": bucket}
            if client_region != "us-east-1":
                params["CreateBucketConfiguration"] = {"LocationConstraint": client_region}
            s3.create_bucket(**params)
            return
        # If we get 403, the bucket likely exists but you don't own it
        if code in ("403", "AccessDenied"):
            raise SystemExit(
                f"Bucket '{bucket}' exists but you do not have access. "
                "Set BUCKET to a globally-unique bucket name you own."
            )
        # Otherwise, re-raise unexpected errors
        raise

def upload(bucket):
    with open(LOCAL_UPLOAD, "w") as f:
        f.write("Hello from boto3 S3!\n")
    s3.upload_file(LOCAL_UPLOAD, bucket, KEY)
    print(f"Uploaded to s3://{bucket}/{KEY}")

def download(bucket):
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    out = os.path.join(DOWNLOAD_DIR, "downloaded.txt")
    s3.download_file(bucket, KEY, out)
    print(f"Downloaded to {out}")

if __name__ == "__main__":
    if not BUCKET:
        raise SystemExit("Set BUCKET environment variable to an existing bucket you own.")
    ensure_bucket(BUCKET)
    upload(BUCKET)
    list_objects(BUCKET)
    download(BUCKET)
```

## Implementing S3 File Download and Verification

Excellent progress with filtering and formatting your S3 object listings! Now it's time to complete the full S3 workflow by implementing the final piece: downloading files and verifying their contents.

You have a working script that creates buckets, uploads files, and lists contents, but the download function is incomplete. It sets up the directory structure and file path, but it is missing the actual download operation and content verification.

Your task is to complete the download function by adding:

* The S3 download operation using `s3.download_file()` with the correct parameters
* File reading logic to open and display the downloaded content

This will demonstrate the complete upload-download-verify cycle and help you understand how to confirm that your S3 operations have worked correctly.

```python
import os, boto3
from botocore.exceptions import ClientError

s3 = boto3.client('s3')

import uuid

BUCKET = os.environ.get("BUCKET") or f"aws-s3-practice-bucket-{str(uuid.uuid4())[:8]}"
KEY = "examples/sample.txt"
LOCAL_UPLOAD = "sample.txt"
DOWNLOAD_DIR = "downloads"

def download(bucket):
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    out = os.path.join(DOWNLOAD_DIR, "downloaded.txt")
    # TODO: Add the s3.download_file() call with bucket, KEY, and out parameters
    print(f"Downloaded to {out}")
    # TODO: Open the downloaded file, read its content, and print it to verify the download worked

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        return
    except ClientError as e:
        code = (e.response.get("Error") or {}).get("Code")
        # If the bucket does NOT exist, create it (in the client's region)
        if code in ("404", "NoSuchBucket"):
            client_region = s3.meta.region_name or boto3.Session().region_name or "us-east-1"
            params = {"Bucket": bucket}
            if client_region != "us-east-1":
                params["CreateBucketConfiguration"] = {"LocationConstraint": client_region}
            s3.create_bucket(**params)
            return
        # If we get 403, the bucket likely exists but you don't own it
        if code in ("403", "AccessDenied"):
            raise SystemExit(
                f"Bucket '{bucket}' exists but you do not have access. "
                "Set BUCKET to a globally-unique bucket name you own."
            )
        # Otherwise, re-raise unexpected errors
        raise

def upload(bucket):
    with open(LOCAL_UPLOAD, "w") as f:
        f.write("Hello from boto3 S3!\n")
    s3.upload_file(LOCAL_UPLOAD, bucket, KEY)
    print(f"Uploaded to s3://{bucket}/{KEY}")

def list_objects(bucket):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix="examples/")
    for obj in resp.get("Contents", []):
        print(obj["Key"], obj["Size"])

if __name__ == "__main__":
    if not BUCKET:
        raise SystemExit("Set BUCKET environment variable to an existing bucket you own.")
    ensure_bucket(BUCKET)
    upload(BUCKET)
    list_objects(BUCKET)
    download(BUCKET)
```

Here is the completed code with the download call and content verification added:

```python
import os, boto3
from botocore.exceptions import ClientError

s3 = boto3.client('s3')

import uuid

BUCKET = os.environ.get("BUCKET") or f"aws-s3-practice-bucket-{str(uuid.uuid4())[:8]}"
KEY = "examples/sample.txt"
LOCAL_UPLOAD = "sample.txt"
DOWNLOAD_DIR = "downloads"

def download(bucket):
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    out = os.path.join(DOWNLOAD_DIR, "downloaded.txt")
    s3.download_file(bucket, KEY, out)
    print(f"Downloaded to {out}")
    with open(out, "r") as f:
        content = f.read()
        print(f"Content: {content}")

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        return
    except ClientError as e:
        code = (e.response.get("Error") or {}).get("Code")
        # If the bucket does NOT exist, create it (in the client's region)
        if code in ("404", "NoSuchBucket"):
            client_region = s3.meta.region_name or boto3.Session().region_name or "us-east-1"
            params = {"Bucket": bucket}
            if client_region != "us-east-1":
                params["CreateBucketConfiguration"] = {"LocationConstraint": client_region}
            s3.create_bucket(**params)
            return
        # If we get 403, the bucket likely exists but you don't own it
        if code in ("403", "AccessDenied"):
            raise SystemExit(
                f"Bucket '{bucket}' exists but you do not have access. "
                "Set BUCKET to a globally-unique bucket name you own."
            )
        # Otherwise, re-raise unexpected errors
        raise

def upload(bucket):
    with open(LOCAL_UPLOAD, "w") as f:
        f.write("Hello from boto3 S3!\n")
    s3.upload_file(LOCAL_UPLOAD, bucket, KEY)
    print(f"Uploaded to s3://{bucket}/{KEY}")

def list_objects(bucket):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix="examples/")
    for obj in resp.get("Contents", []):
        print(obj["Key"], obj["Size"])

if __name__ == "__main__":
    if not BUCKET:
        raise SystemExit("Set BUCKET environment variable to an existing bucket you own.")
    ensure_bucket(BUCKET)
    upload(BUCKET)
    list_objects(BUCKET)
    download(BUCKET)
```